#  Spam Detector — Logistic Regression
**Dhruti Movaliya | movaliyadhruti3@gmail.com**

---
### Features used to classify an email:
- `spam_words` → count of trigger words (free, win, click…)
- `exclamations` → count of '!' marks
- `links` → count of hyperlinks / URLs

**Label:** `1 = SPAM` | `0 = NOT SPAM (Ham)`

In [ ]:
# ──────────────────────────────────────────────────────────────
# IMPORTS
# ──────────────────────────────────────────────────────────────
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print(" Libraries imported successfully!")

## CELL 2 — Dataset
Setting up the email features and labels.

In [ ]:
# Features: [spam_words, exclamations, links]
X = [
    [20, 10, 5],  # 1 - SPAM
    [ 2,  1, 0],  # 0 - Ham
    [18,  8, 3],  # 1 - SPAM
    [ 1,  0, 1],  # 0 - Ham
    [15,  6, 4],  # 1 - SPAM
    [ 3,  2, 0],  # 0 - Ham
    [22, 12, 6],  # 1 - SPAM
    [ 0,  0, 0],  # 0 - Ham
    [17,  9, 4],  # 1 - SPAM
    [ 1,  1, 0],  # 0 - Ham
    [25, 14, 7],  # 1 - SPAM
    [ 0,  1, 0],  # 0 - Ham
    [19, 11, 5],  # 1 - SPAM
    [ 2,  0, 0],  # 0 - Ham
    [16,  7, 3],  # 1 - SPAM
    [ 3,  1, 1],  # 0 - Ham
    [21, 13, 6],  # 1 - SPAM
    [ 1,  0, 0],  # 0 - Ham
    [14,  5, 3],  # 1 - SPAM
    [ 4,  2, 1],  # 0 - Ham
]

y = [1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0]

X = np.array(X)
y = np.array(y)

print(f"Total emails: {len(y)}")
print(f"Spam (1)    : {sum(y)}")
print(f"Ham (0)     : {len(y) - sum(y)}")
print(f"Feature shape: {X.shape}")

## CELL 3 — Train Model & Predict
Training the Logistic Regression model using `train_test_split`.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

print(f"Train samples : {len(X_train)}")
print(f"Test samples  : {len(X_test)}\n")

model = LogisticRegression()
model.fit(X_train, y_train)          # TRAIN
print("✔ Model trained successfully!\n")

y_pred       = model.predict(X_test)            # PREDICT
y_pred_proba = model.predict_proba(X_test)      # PROBABILITIES

print("Classification Report:")
print("-" * 53)
print(classification_report(
    y_test, y_pred,
    labels=[0, 1],
    target_names=["Not Spam", "Spam"],
    zero_division=0
))

## Prediction Results — Test Emails

In [ ]:
header = f"{'#':<5}{'SpamWds':<10}{'Excl':<8}{'Links':<8}{'Actual':<12}{'Predicted':<14}{'P(Spam)':<10}{'Bar'}"
print(header)
print("─" * 85)

for i in range(len(X_test)):
    actual    = "SPAM" if y_test[i] == 1 else "HAM "
    predicted = "SPAM" if y_pred[i] == 1 else "HAM "
    p_spam    = y_pred_proba[i][1]
    bar_len   = int(p_spam * 20)
    bar       = f"{'█' * bar_len}{'░' * (20 - bar_len)}"
    correct   = "✔" if y_test[i] == y_pred[i] else "✘"
    print(f"{i+1:<5}{X_test[i][0]:<10}{X_test[i][1]:<8}{X_test[i][2]:<8}"\
          f"{actual:<12}{predicted:<14}{p_spam:<10.3f}{bar}  {correct}")

## Visualizing the Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
acc = accuracy_score(y_test, y_pred)

plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Not Spam (0)', 'Spam (1)'],
            yticklabels=['Not Spam (0)', 'Spam (1)'])
plt.ylabel('Actual Label')
plt.xlabel('Predicted Label')
plt.title(f'Confusion Matrix (Accuracy: {acc*100:.1f}%)')
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"✔ TN = Correctly identified Ham: {tn}")
print(f"✔ TP = Correctly caught Spam: {tp}")
print(f"✘ FP = Ham wrongly flagged as Spam: {fp}")
print(f"✘ FN = Spam that slipped through: {fn}")

## Feature Importance (Model Weights)

In [ ]:
features = ["spam_words", "exclamations", "links"]
weights  = model.coef_[0]
print(f"{'Feature':<16}  Weight    Impact")
print("─" * 40)
for f, w in zip(features, weights):
    bar_len = int(abs(w) * 5)
    bar     = f"{'▮' * bar_len}"
    sign    = "+" if w > 0 else ""
    print(f"{f:<16}  {sign}{w:>6.3f}    {bar}")

print("\nTip: Higher positive weight = stronger spam indicator")

## Predict Your Own Email 

In [ ]:
def predict_email(spam_words, exclamations, links, label=""):
    features  = np.array([[spam_words, exclamations, links]])
    result    = model.predict(features)[0]
    prob      = model.predict_proba(features)[0]
    p_spam    = prob[1]
    bar_len   = int(p_spam * 30)
    bar       = f"{'█' * bar_len}{'░' * (30 - bar_len)}"

    if result == 1:
        verdict = " SPAM"
    else:
        verdict = " NOT SPAM (Ham)"

    tag = f" ({label})" if label else ""
    print(f"\n📧 Email{tag}: spam_words={spam_words}, exclamations={exclamations}, links={links}")
    print(f"   Verdict   : {verdict}")
    print(f"   P(Spam)   : {p_spam:.3f}  [{bar}]  {p_spam*100:.1f}%")

predict_email(20, 10, 5,  "obvious spam")
predict_email(1,  0,  0,  "obvious ham")
predict_email(10, 5,  2,  "borderline")
predict_email(8,  3,  1,  "your guess?")